# 02 Groundwater Monitoring
Developed by: Lilly Jones, PhD, Daear Consulting LLC                                                                               
Developed for: Oglala Lakota College                                                                                            
Funding: This material was developed as part of a project funded by the USDA National Institute of Food and Agriculture (NIFA).                       
Project role: Daear Consulting LLC developed the geospatial code, workflows, documentation, and instructional materials under contract to Oglala Lakota College.                                                                           
License: Apache License 2.0 (code; review of other materials is pending)
Data Sources: USGS NWIS public groundwater observations and published inventory context     

## Data Sovereignty and Governance (draft under review)
This repository contains workflows developed for use in support of Oglala Lakota College and Oglala Sioux Tribe–related research, education, and data activities. Public availability of code or documentation does not imply that Tribal data, knowledge, or derived information are open or unrestricted. Use of Tribal data and knowledge remains subject to applicable Tribal governance, permissions, protocols, and data sovereignty requirements.                                                                                              
  
## Why Groundwater First
On Pine Ridge, groundwater is the primary water source. 
Surface streams are seasonal and often dry outside spring runoff. 
The communities, livestock operations, and bison herds that sustain life 
on these lands depend on what comes out of the ground.

The principal aquifer beneath Pine Ridge is the Arikaree
aquifer (a formation of the Ogallala/High Plains Aquifer system). It is a finite
resource: recharge is slow (decades to centuries), extraction is immediate.
Climate stress such as increased evaporation, less snowpack recharge, and longer droughts
is reducing recharge while demand is stable or increasing.

## Data Approach
This notebook uses three complementary data sources:
**USGS monitoring locations and field measurements:** A sparse public
monitoring network around the study area. These sites can have long records,
but they are not a complete well inventory.

**Historical well inventories:** Carter and Heakin (2007) documents wells
and springs in Pine Ridge and Bennett County. Inventory wells must not be
presented as currently monitored wells or current water-level observations.

**Governance boundary:** This public teaching path does not load OST-controlled data. Any future use requires a separately approved process.

## Research Questions
- What do long-term USGS well records near Pine Ridge show?
- Is there a declining trend in water levels?
- Where are the monitoring dead zones (areas with no well data at all)?
- What additional evidence or local expertise would be needed to understand conditions not represented by the selected public sites?

## Learning Objectives

By the end of this notebook, learners will be able to:

- inspect record length, missingness, and sampling frequency before fitting a trend
- interpret the direction and uncertainty of a depth-to-water trend
- distinguish absence of public monitoring evidence from evidence about groundwater conditions

## Prerequisites and Timing

Allow approximately 75–100 minutes. Before beginning, activate the repository environment, read the series governance statement, and complete the preceding notebook where applicable. Work in pairs and rotate analyst, data-steward, skeptic, and documentarian roles.

## Governance Checkpoint

This notebook uses public environmental data describing Oglala Lakota lands and waters. Public availability does not establish permission for every reuse or interpretation. Do not add OST-controlled data, sensitive locations, or community knowledge. Results are educational and screening-level pending OLC/OST review.

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
import requests
from datetime import datetime

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from scipy import stats

from src.constants import (
    CRS_GEOGRAPHIC, CRS_PROJECTED,
    OUTPUTS_DIR, FIGURES_DIR,
)
from src.config import load_config, streamflow_site_ids, streamflow_site_names

CONFIG = load_config()
STUDY_BBOX = tuple(CONFIG["study_area"]["hydrologic_context_bbox"])
STUDY_NAMES = [CONFIG["study_area"]["people"]]
STUDY_CENTROIDS = {CONFIG["study_area"]["people"]: CONFIG["study_area"]["centroid"]}
PINE_RIDGE_STREAMGAGES = {
    site["name"]: str(site["id"]) for site in CONFIG["usgs_streamflow_sites"]
}

from src.loaders import (
    load_tribal_boundaries,
    load_usgs_groundwater_sites,
    load_usgs_groundwater_levels,
    load_carter_2007_inventory,
)
from src.indicators import (
    compute_groundwater_trend,
    classify_groundwater_status,
    groundwater_percentile_rank,
    theilsen_trend,
)
from src.sovereignty import print_data_acknowledgment, generate_citations

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
%matplotlib inline

def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# print(f"Repo root : {REPO_ROOT}")
print(f"Analysis  : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("Imports complete.")

In [ ]:
# Print data sovereignty statement at the top of every notebook
print_data_acknowledgment(
    source_keys=["census_aiannh", "usgs_nwis_groundwater", "tribal_groundwater"]
)

## Load Boundaries and USGS Well Sites

In [ ]:
# Load from notebook 01 output if available
boundaries_path = OUTPUTS_DIR/"pine_ridge_census_boundary.geojson"
if boundaries_path.exists():
    study_boundary = gpd.read_file(boundaries_path)
    print(f"Boundary features loaded from notebook 01: {len(study_boundary)}")
else:
    study_boundary = load_tribal_boundaries()
    print(f"Boundary features downloaded: {len(study_boundary)}")

primary = study_boundary[study_boundary["common_name"].isin(STUDY_NAMES)]

In [ ]:
# Discover USGS groundwater monitoring wells in the study area.
# This is deliberately separate from the historical Carter (2007) inventory.
gw_sites = load_usgs_groundwater_sites(bbox=STUDY_BBOX)
carter_inventory = load_carter_2007_inventory()

print(f"USGS groundwater sites found: {len(gw_sites)}")
print(f"Carter 2007 inventory features: {len(carter_inventory)} (not current monitoring)")
if not gw_sites.empty and "site_no" in gw_sites.columns:
    print("\nSite IDs (use these in config.yaml > usgs_groundwater_sites):")
    cols = [c for c in ["site_no", "station_nm", "dec_lat_va",
                        "dec_long_va", "begin_date", "end_date"]
            if c in gw_sites.columns]
    print(gw_sites[cols].head(15).to_string(index=False))

In [ ]:
# Filter to wells with long records that are the most useful for trend analysis
# Prioritize wells nearest to Pine Ridge centroids

if not gw_sites.empty:
    primary_proj  = primary.to_crs(CRS_PROJECTED)
    sites_proj    = gw_sites.to_crs(CRS_PROJECTED)

    candidate_sites = []
    for _, nation in primary_proj.iterrows():
        centroid = nation.geometry.centroid
        dists    = sites_proj.geometry.distance(centroid) / 1000
        nearby   = gw_sites[(dists <= 100).values].copy()
        nearby["dist_km"]    = dists[(dists <= 100).values].round(1).values
        nearby["nation"]     = nation["common_name"]
        candidate_sites.append(nearby)

    candidates = pd.concat(candidate_sites, ignore_index=True).drop_duplicates(
        subset="site_no"
    )

    # Check record length if begin_date available
    if "begin_date" in candidates.columns:
        candidates["begin_date"] = pd.to_datetime(
            candidates["begin_date"], errors="coerce"
        )
        candidates["record_years"] = (
            (pd.Timestamp.now() - candidates["begin_date"]).dt.days / 365
        ).round(1)
        long_record = candidates[candidates["record_years"] >= 10].copy()
        print(f"Wells within 100 km of primary Nations: {len(candidates)}")
        print(f"Wells with ≥ 10 years of record: {len(long_record)}")
    else:
        long_record = candidates.copy()
        print(f"Candidate wells: {len(candidates):,}")
        print("Next cell filters these by recent USGS groundwater-level records.")
else:
    long_record = pd.DataFrame()
    print("No USGS wells found. This is a monitoring gap finding.")

In [ ]:
# Select candidates with recent USGS depth-to-water field measurements.
# Metadata is queried once for the area, then joined to the nearby candidates.
FIELD_METADATA_URL = (
    "https://api.waterdata.usgs.gov/ogcapi/v0/collections/"
    "field-measurements-metadata/items"
)
RECENT_SINCE = "2020-01-01T00:00:00Z"

def fetch_ogc_features(url, params):
    features = []
    while url:
        response = requests.get(url, params=params, timeout=120)
        response.raise_for_status()
        payload = response.json()
        features.extend(payload.get("features", []))
        url = next((link.get("href") for link in payload.get("links", [])
                    if link.get("rel") == "next"), None)
        params = {}  # next link already includes its query parameters
    return features

recent_wells = pd.DataFrame()
if not long_record.empty:
    metadata_features = fetch_ogc_features(
        FIELD_METADATA_URL,
        {
            "bbox": ",".join(map(str, STUDY_BBOX)),
            "parameter_code": "72019",  # depth to water below land surface
            "end": f"{RECENT_SINCE}/..",
            "limit": 1000,
            "f": "json",
        },
    )
    field_metadata = pd.DataFrame([f.get("properties", {}) for f in metadata_features])
    if not field_metadata.empty:
        field_metadata["end"] = pd.to_datetime(field_metadata["end"], errors="coerce", utc=True)
        recent_wells = (
            long_record.merge(
                field_metadata[["monitoring_location_id", "begin", "end", "parameter_name"]],
                on="monitoring_location_id", how="inner",
            )
            .sort_values(["end", "dist_km"], ascending=[False, True])
            .drop_duplicates(subset="site_no")
            .reset_index(drop=True)
        )

if recent_wells.empty:
    print(f"No nearby USGS depth-to-water records since {RECENT_SINCE[:4]}.")
else:
    print(f"Recent groundwater-level candidates: {len(recent_wells):,}")
    print(recent_wells[["site_no", "station_nm", "end", "dist_km"]].head(10).to_string(index=False))
    long_record = recent_wells  # consumed by the time-series cell below


## Fetch USGS Groundwater Level Time Series

In [ ]:
# Fetch water level records for up to 5 wells
MAX_WELLS   = 5
gwl_parts   = []
gwl_failed  = []

wells_to_fetch = long_record.head(MAX_WELLS) if not long_record.empty else pd.DataFrame()

for _, site in wells_to_fetch.iterrows():
    site_no   = site["site_no"]
    site_name = site.get("station_nm", site_no)
    try:
        df = load_usgs_groundwater_levels(site_no=site_no)
        if df.empty:
            print(f"  {site_name}: no data")
            continue
        df["site_name"] = str(site_name)[:50]
        df["nation"]    = site.get("nation", "Unknown")
        gwl_parts.append(df)
        print(f"  {site_name}: {len(df)} measurements "
              f"({df['date'].min().year}–{df['date'].max().year})")
    except Exception as e:
        gwl_failed.append(site_no)
        print(f"  {site_name}: failed — {e}")

if gwl_parts:
    gwl_df = pd.concat(gwl_parts, ignore_index=True)
    gwl_df["year"]  = gwl_df["date"].dt.year
    gwl_df["month"] = gwl_df["date"].dt.month
    print(f"\nTotal USGS groundwater records: {len(gwl_df):,}")
else:
    gwl_df = pd.DataFrame()
    print("\nNo USGS groundwater level data available.")
    print("Coverage gap near Tribal lands is itself a critical finding.")

## Trend Analysis

In [ ]:
# Trend analysis for USGS wells
trend_records = []

if not gwl_df.empty:
    for site_no, grp in gwl_df.groupby("site_no"):
        annual = grp.groupby("year")["water_level_ft"].median().reset_index()
        if len(annual) < 5:
            continue
        trend = theilsen_trend(
            values=annual["water_level_ft"].values,
            years=annual["year"].values,
        )
        trend["site_no"]   = site_no
        trend["site_name"] = grp["site_name"].iloc[0]
        trend["nation"]    = grp["nation"].iloc[0]
        trend["n_years"]   = len(annual)
        # Positive slope = depth increasing = water table falling
        trend["direction"] = "Declining" if trend["slope"] > 0 else "Rising/Stable"
        trend_records.append(trend)

if trend_records:
    trend_df = pd.DataFrame(trend_records)
    print("GROUNDWATER LEVEL TRENDS (Theil-Sen slope)")
    print("Positive slope = depth to water INCREASING = water table FALLING")
    print(
        trend_df[["site_name", "nation", "n_years",
                  "slope_per_decade", "direction", "p_value", "significant"]]
        .sort_values("slope_per_decade", ascending=False)
        .to_string(index=False)
    )
else:
    trend_df = pd.DataFrame()
    print("Insufficient USGS well data for trend analysis.")
    print("The selected public record has limited coverage; causes and implications require additional evidence.")

## Visualizations

In [ ]:
# USGS well time series
if not gwl_df.empty:
    sites = gwl_df["site_no"].unique()
    n     = len(sites)
    ncols = min(2, n)
    nrows = (n + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(13, nrows * 4))
    axes = np.array(axes).flatten()

    for i, site_no in enumerate(sites):
        ax   = axes[i]
        grp  = gwl_df[gwl_df["site_no"] == site_no].sort_values("date")
        name = grp["site_name"].iloc[0]
        nation = grp["nation"].iloc[0]

        ax.scatter(grp["date"], grp["water_level_ft"],
                   color="#1A5276", s=6, alpha=0.4)

        # Annual median
        annual = grp.groupby("year")["water_level_ft"].median().reset_index()
        annual["date_mid"] = pd.to_datetime(annual["year"].astype(str) + "-07-01")
        ax.plot(annual["date_mid"], annual["water_level_ft"],
                color="#1A5276", linewidth=2, label="Annual median")

        # Trend
        if not trend_df.empty:
            tr = trend_df[trend_df["site_no"] == site_no]
            if not tr.empty:
                slope = tr["slope"].iloc[0]
                yrs   = annual["year"].values.astype(float)
                t_line = slope * (yrs - yrs.mean()) + annual["water_level_ft"].mean()
                ax.plot(annual["date_mid"], t_line, color="#C0392B",
                        linewidth=1.5, linestyle="--",
                        label=f"Trend: {tr['slope_per_decade'].iloc[0]:+.2f} ft/decade")

        # deeper water = down 
        ax.invert_yaxis()  
        ax.set_ylabel("Depth to water (ft below surface)", fontsize=8)
        ax.set_title(f"{name[:45]}\n{nation}", fontsize=8, fontweight="bold")
        ax.legend(fontsize=7)
        despine(ax)

    for ax in axes[n:]:
        ax.set_visible(False)

    plt.suptitle(
        "USGS Groundwater Levels Near Pine Ridge\n"
        "Y-axis inverted: up = shallower water | Red dashed = Theil-Sen trend",
        fontsize=11, fontweight="bold",
    )
    plt.tight_layout()
    try:
        fig.savefig(FIGURES_DIR/"02_groundwater_timeseries.png",
                    dpi=150, bbox_inches="tight")
    except Exception:
        pass
    plt.show()
else:
    print("No USGS groundwater level data to plot.")
    print("The monitoring gap map (Figure 01) shows why.")

## Exports

In [ ]:
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

if not gwl_df.empty:
    gwl_df.to_csv(OUTPUTS_DIR/"usgs_groundwater_levels.csv", index=False)
    print("Exported to outputs/usgs_groundwater_levels.csv")

if not trend_df.empty:
    trend_df.to_csv(OUTPUTS_DIR/"groundwater_trends.csv", index=False)
    print("Exported to outputs/groundwater_trends.csv")

# Note: Tribal groundwater data is NOT exported to outputs/ 
# it stays in data/raw/ under Tribal governance

In [ ]:
print(generate_citations(["usgs_nwis_groundwater"]))

## Learner Checkpoint

Select one well. Explain its period of record, the sign convention for depth to water, and whether the available observations justify a trend statement.

## Interpretation Protocol

Before writing a conclusion, separate:

1. **Observation:** what the computed public data show, including unit, period, spatial scope, and missingness.
2. **Interpretation:** a plausible explanation, stated with uncertainty.
3. **Additional evidence:** literature, local monitoring, expertise, or validation needed to evaluate that explanation.
4. **Decision authority:** who is authorized to approve publication, thresholds, or management action.

Do not convert monitoring absence, association, a screening flag, or scenario output into a causal, regulatory, health, policy, or community conclusion.

## Contribution Activity

Add or improve one public-site description, unit note, missing-data warning, or trend limitation. Do not add OST-controlled data. Review the change with a partner and record what became clearer or more defensible.

## Evidence Record and Next Step

Record one regenerated result, its source and scope, one transformation, one limitation, and one question requiring more evidence or local knowledge.

Notebook 03 examines seasonal streamflow and screening-level reliability measures for configured public streamgages.